# Few-shot, one site at a time

Scores for the fold-3 runs that train on labelled tiles from one site (or from every training site) and then test on the sites the fold holds out.

Run this notebook from the repo root or from `notebooks/`. It only reads `lightning_logs/` and writes PNGs to `notebooks/few_shot_site_figures/`.

**What each run is.** U-Net, MiT-B2, 512 px crops, 30 epochs, validation after epochs 10, 20, and 30. `k` is labelled tiles per species at the chosen site. `bg` is extra empty tiles from that same site. The tile choice is `max_px_site_pheno`. Seeds are 42 and 43. A site name means every flight of that site. `all` means every training site in the fold, with the same `k` spread across them.

**Two scores.** Same-site is the val folders of the training site: other tiles of the same flights, never used for training. Held-out is the fold-3 test: Basel_2, Dornach_1, Rivera_1, and Waltenschwil_1. The number is mean species F1 (`F1-avg-wo0`), background left out. The checkpoint is the best of the three validation passes. A run that has not finished its test is listed and left out of the figures.

Earlier few-shot runs that were scored on the Sion debug split stay in `few_shot_presentation.ipynb`. This notebook does not plot those.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter

cwd = Path.cwd()
if (cwd / "lightning_logs").is_dir():
    repo = cwd
elif (cwd.parent / "lightning_logs").is_dir():
    repo = cwd.parent
else:
    raise FileNotFoundError(
        "Cannot find lightning_logs/. Start the kernel in the repo root or in notebooks/."
    )
LOG_ROOT = repo / "lightning_logs"
FIG_DIR = repo / "notebooks" / "few_shot_site_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

RUN_RE = re.compile(
    r"_neophytes_split_cv3_train_512_model_neophytes_mit_b2_s(?P<seed>42|43)"
    r"(?:_(?P<tag>site[A-Za-z0-9_]+|fl\d+|fl_[A-Za-z0-9_-]+))?"
    r"_t(?P<k>\d+)_tbg(?P<bg>\d+)$"
)
SPECIES = [
    "A. altissima",
    "R. typhina",
    "B. davidii",
    "R. japonica",
    "B. orientalis",
    "S. inaequidens",
]
SITE_ORDER = ["Basel_4", "Olten_1", "Ardon_1", "Arlesheim_1", "Dornach_2", "all"]
HELD_OUT = ["Basel_2", "Dornach_1", "Rivera_1", "Waltenschwil_1"]
SITE_COLOR = {
    "Basel_4": "#0072B2",
    "Olten_1": "#E69F00",
    "Ardon_1": "#009E73",
    "Arlesheim_1": "#CC79A7",
    "Dornach_2": "#D55E00",
    "all": "#000000",
}
SPECIES_COLOR = {
    "A. altissima": "#dd60f6",
    "R. typhina": "#b76e44",
    "B. davidii": "#dce14c",
    "R. japonica": "#305dc6",
    "B. orientalis": "#e2256a",
    "S. inaequidens": "#50e3c2",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "savefig.dpi": 160,
})


def load_json(path):
    return json.loads(path.read_text(), parse_constant=lambda token: float(token))


def as_site(name):
    """``Basel_4_1`` (a flight) -> ``Basel_4``. A site key is left as it is."""
    parts = str(name).split("_")
    if len(parts) >= 3 and parts[-1].isdigit() and parts[-2].isdigit():
        return "_".join(parts[:-1])
    return str(name)


def parse_train_site(text):
    match = re.search(r"data\.train_site=(\[[^\]]+\]|\S+)", text)
    if not match:
        return "all"
    raw = match.group(1).strip("[]")
    groups = []
    for key in raw.split(","):
        key = key.strip()
        if key and key not in groups:
            groups.append(as_site(key))
    if not groups:
        return "all"
    if len(groups) == 1:
        return groups[0]
    return "+".join(groups)


def site_from_run(run, tag):
    overrides = sorted(run.glob("hydra/*/.hydra/overrides.yaml"))
    if overrides:
        return parse_train_site(overrides[-1].read_text())
    if tag and tag.startswith("site"):
        return tag[len("site"):]
    if tag:
        return tag
    return "all"


def cv3_tests(run):
    found = []
    for path in sorted(run.glob("test_*")):
        if "np_cv3" not in path.name or "np_debug" in path.name:
            continue
        if (path / "quantitative" / "metrics_semseg.json").is_file():
            found.append(path)
    return found


def species_scores(block):
    names = block.get("class_names") or []
    scores = block.get("F1") or []
    return {
        name: scores[i]
        for i, name in enumerate(names)
        if name in SPECIES and i < len(scores)
    }


def expected_rows():
    rows = []
    for site in SITE_ORDER[:-1]:
        for k in (1, 3, 10):
            for seed in (42, 43):
                rows.append({"site": site, "k": k, "bg": 100, "seed": seed})
    for seed in (42, 43):
        rows.append({"site": "Basel_4", "k": 10, "bg": 10, "seed": seed})
        rows.append({"site": "Basel_4", "k": 10, "bg": 1000, "seed": seed})
        rows.append({"site": "all", "k": 10, "bg": 100, "seed": seed})
    return pd.DataFrame(rows)


def load_runs(log_root):
    records = []
    flights = []
    for run in sorted(p for p in log_root.iterdir() if p.is_dir()):
        match = RUN_RE.search(run.name)
        if not match:
            continue
        debug = [p for p in run.glob("test_*") if "np_debug" in p.name]
        tests = cv3_tests(run)
        if debug and not tests:
            continue
        site = site_from_run(run, match.group("tag"))
        row = {
            "run": run.name,
            "site": site,
            "k": int(match.group("k")),
            "bg": int(match.group("bg")),
            "seed": int(match.group("seed")),
            "tested": bool(tests),
        }
        if not tests:
            records.append(row)
            continue
        test = tests[-1]
        quant = test / "quantitative"
        semseg = load_json(quant / "metrics_semseg.json")
        split_path = quant / "metrics_site_split.json"
        split = load_json(split_path) if split_path.is_file() else None
        held_block = (split or {}).get("held_out") or semseg
        same_block = (split or {}).get("same_site")
        row.update(
            test=test.name,
            held_f1=held_block.get("F1-avg-wo0"),
            same_f1=None if same_block is None else same_block.get("F1-avg-wo0"),
            held_tiles=None if split is None else held_block.get("tiles"),
            same_tiles=None if same_block is None else same_block.get("tiles"),
        )
        for species, value in species_scores(held_block).items():
            row[f"held_{species}"] = value
        if same_block is not None:
            for species, value in species_scores(same_block).items():
                row[f"same_{species}"] = value
        site_path = quant / "metrics_semseg_site.json"
        if site_path.is_file():
            per_flight = load_json(site_path)
            for flight, block in per_flight.items():
                group = as_site(flight)
                if group not in HELD_OUT:
                    continue
                flights.append({
                    "run": run.name,
                    "site": site,
                    "k": row["k"],
                    "bg": row["bg"],
                    "seed": row["seed"],
                    "flight": flight,
                    "held_site": group,
                    "f1": block.get("F1-avg-wo0"),
                })
        records.append(row)
    frame = pd.DataFrame.from_records(records)
    if frame.empty:
        frame = pd.DataFrame(columns=["run", "site", "k", "bg", "seed", "tested"])
    else:
        frame = frame.sort_values(["tested", "site", "k", "bg", "seed", "run"])
        # A repeated queue of the same setting: keep the tested copy, else the newest name.
        frame = frame.sort_values(["tested", "run"]).drop_duplicates(
            ["site", "k", "bg", "seed"], keep="last"
        )
    flight_frame = pd.DataFrame.from_records(flights)
    frame = frame.reset_index(drop=True)
    if not flight_frame.empty:
        flight_frame = flight_frame[flight_frame["run"].isin(set(frame["run"]))].reset_index(drop=True)
    return frame, flight_frame


def save(fig, name):
    fig.savefig(FIG_DIR / name, bbox_inches="tight")
    plt.show()


def finish_score_axis(ax, xlabel):
    ax.set_ylim(0, 1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("mean species F1")
    ax.set_xticks([1, 3, 10])
    ax.grid(axis="y", color="#eeeeee")


def plot_lines(ax, part, y):
    for site in SITE_ORDER:
        group = part[part["site"] == site]
        if group.empty or y not in group or group[y].isna().all():
            continue
        color = SITE_COLOR.get(site, "#333333")
        mean = group.groupby("k", as_index=False)[y].mean().sort_values("k")
        ax.plot(mean["k"], mean[y], color=color, lw=2, marker="o", ms=6, label=site, zorder=2)
        ax.scatter(group["k"], group[y], color=color, s=18, alpha=0.45, zorder=3)


runs, flights = load_runs(LOG_ROOT)
plan = expected_rows()
status = plan.merge(
    runs[["site", "k", "bg", "seed", "tested", "run"]],
    on=["site", "k", "bg", "seed"],
    how="left",
)
status["state"] = np.where(
    status["tested"] == True, "tested",
    np.where(status["run"].notna(), "waiting for test", "not started"),
)
scored = runs[runs["tested"] == True].copy() if "tested" in runs else runs.iloc[0:0]
print(
    f"{int((status['state'] == 'tested').sum())} tested, "
    f"{int((status['state'] == 'waiting for test').sum())} waiting for test, "
    f"{int((status['state'] == 'not started').sum())} not started "
    f"(of {len(status)})"
)
status

## Scores

Same-site is how the model does on the rest of the site it trained on. Held-out is the four sites it never trained on. A blank species was not labelled on that split, so it is not part of the average. `all` has no same-site column: its validation covers every training site, and the comparison that matches the other rows is the held-out score.

In [ ]:
held_cols = [f"held_{name}" for name in SPECIES]
columns = ["site", "k", "bg", "seed", "same_f1", "held_f1", *held_cols]
if scored.empty:
    print("No fold-3 test yet. The table above is the queue.")
else:
    view = scored.reindex(columns=columns).copy()
    for column in view.columns[4:]:
        view[column] = pd.to_numeric(view[column], errors="coerce")
        view[column] = view[column].map(lambda value: None if pd.isna(value) else round(float(value), 3))
    view = view.sort_values(["site", "bg", "k", "seed"]).reset_index(drop=True)
view if not scored.empty else None

## Tiles per species

Background is held at 100. The line is the mean of the two seeds. The faint dots are seed 42 and seed 43. The left panel is the training site. The right panel is the held-out sites.

In [ ]:
part = scored[scored["bg"] == 100] if not scored.empty else scored
if part.empty:
    print("No tested run with 100 background tiles yet.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    plot_lines(axes[0], part, "same_f1")
    finish_score_axis(axes[0], "labelled tiles per species")
    axes[0].set_title("same site")
    plot_lines(axes[1], part, "held_f1")
    finish_score_axis(axes[1], "labelled tiles per species")
    axes[1].set_title("held-out sites")
    axes[1].set_ylabel("")
    handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=6, frameon=False, bbox_to_anchor=(0.5, 1.08))
    fig.tight_layout()
    save(fig, "k_same_vs_heldout.png")

## Each species on the held-out sites

Same runs as the panel above: 100 background tiles. Each line is a training site. A species with no labelled pixels on the held-out sites stays blank.

In [ ]:
part = scored[scored["bg"] == 100] if not scored.empty else scored
if part.empty:
    print("No tested run with 100 background tiles yet.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(11, 6.2), sharex=True, sharey=True)
    for ax, species in zip(axes.ravel(), SPECIES):
        column = f"held_{species}"
        plot_lines(ax, part, column)
        ax.set_title(species)
        ax.set_ylim(0, 1)
        ax.set_xticks([1, 3, 10])
        ax.grid(axis="y", color="#eeeeee")
        values = pd.to_numeric(part.get(column, pd.Series(dtype=float)), errors="coerce")
        if not np.isfinite(values.to_numpy(dtype=float)).any():
            ax.text(0.5, 0.5, "no labels on this test", transform=ax.transAxes,
                    ha="center", va="center", color="0.45", fontsize=10)
    for ax in axes[1]:
        ax.set_xlabel("labelled tiles per species")
    axes[0, 0].set_ylabel("species F1")
    axes[1, 0].set_ylabel("species F1")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=6, frameon=False, bbox_to_anchor=(0.5, 1.05))
    fig.tight_layout()
    save(fig, "species_heldout_bg100.png")

## Empty tiles, Basel_4 only

Ten labelled tiles per species. Background is 10, 100, or 1000 empty Basel_4 tiles. Filled dots are the held-out sites. Open dots are the rest of Basel_4.

In [ ]:
part = scored[(scored["site"] == "Basel_4") & (scored["k"] == 10)] if not scored.empty else scored
if part.empty:
    print("No tested Basel_4 run at k=10 yet.")
else:
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for column, marker, label in (
        ("held_f1", "o", "held-out sites"),
        ("same_f1", "o", "same site"),
    ):
        group = part.dropna(subset=[column])
        if group.empty:
            continue
        mean = group.groupby("bg", as_index=False)[column].mean().sort_values("bg")
        filled = label.startswith("held")
        ax.plot(mean["bg"], mean[column], color="#0072B2", lw=2, zorder=2)
        ax.scatter(
            group["bg"], group[column],
            facecolors="#0072B2" if filled else "white",
            edgecolors="#0072B2", s=40, label=label, zorder=3,
        )
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(FixedLocator([10, 100, 1000]))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _pos: f"{int(value)}"))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlim(7, 1400)
    ax.set_ylim(0, 1)
    ax.set_xlabel("background tiles")
    ax.set_ylabel("mean species F1")
    ax.grid(axis="y", color="#eeeeee")
    ax.legend(frameon=False)
    fig.tight_layout()
    save(fig, "basel4_background.png")

## Where the held-out score comes from

Ten labelled tiles and 100 background tiles. Each cell is mean species F1 on one held-out site. A site with several flights contributes the unweighted mean of those flights, then the two seeds are averaged. This is a map of where the model fails. The single held-out number in the table above is the pooled score, which gives more weight to the larger sites.

In [ ]:
part = flights[(flights["k"] == 10) & (flights["bg"] == 100)] if not flights.empty else flights
if part.empty:
    print("No per-site scores yet. They appear when a fold-3 test finishes.")
else:
    by_site = part.groupby(["site", "seed", "held_site"], as_index=False)["f1"].mean()
    grid = by_site.groupby(["site", "held_site"], as_index=False)["f1"].mean()
    order = [site for site in SITE_ORDER if site in set(grid["site"])]
    matrix = (
        grid.pivot(index="site", columns="held_site", values="f1")
        .reindex(index=order, columns=HELD_OUT)
    )
    fig, ax = plt.subplots(figsize=(7.2, 0.55 * len(matrix) + 1.6))
    image = ax.imshow(matrix.to_numpy(dtype=float), vmin=0, vmax=1, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set_xlabel("held-out site")
    ax.set_ylabel("training site")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix.iloc[i, j]
            if pd.notna(value):
                ax.text(j, i, f"{value:.2f}", ha="center", va="center", color="white", fontsize=9)
    fig.colorbar(image, ax=ax, label="mean species F1", fraction=0.046)
    fig.tight_layout()
    save(fig, "heldout_site_k10.png")
    matrix.round(3)

## One site or every site

Ten labelled tiles per species and 100 background tiles. Each bar is the mean of the two seeds. The dots are the seeds. `all` spreads those ten tiles across every training site. The other bars keep all ten at one site.

In [ ]:
part = scored[(scored["k"] == 10) & (scored["bg"] == 100)] if not scored.empty else scored
if part.empty:
    print("No tested run at k=10 and 100 background tiles yet.")
else:
    order = [site for site in SITE_ORDER if site in set(part["site"])]
    fig, ax = plt.subplots(figsize=(7.6, 4.2))
    x = np.arange(len(order))
    width = 0.36
    for shift, column, label, color in (
        (-width / 2, "same_f1", "same site", "#56B4E9"),
        (width / 2, "held_f1", "held-out sites", "#0072B2"),
    ):
        means = []
        for site in order:
            values = pd.to_numeric(part.loc[part["site"] == site, column], errors="coerce")
            means.append(np.nanmean(values.to_numpy(dtype=float)) if values.notna().any() else np.nan)
            ax.scatter(
                np.full(values.notna().sum(), order.index(site)) + shift,
                values.dropna(),
                color=color, s=22, zorder=3,
            )
        ax.bar(x + shift, means, width=width, color=color, label=label, zorder=2)
    ax.set_xticks(x, order, rotation=30, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("mean species F1")
    ax.grid(axis="y", color="#eeeeee")
    ax.legend(frameon=False)
    fig.tight_layout()
    save(fig, "one_site_vs_all_k10.png")

## Where, and by stage

The first figure is the list of runs: which site, how many labelled tiles, how many empty tiles, and the two seeds. Filled marks already have a fold-3 test.

The map is in Swiss coordinates. Each card is mean species F1 for 1, 3, and 10 labelled tiles, averaged over the two seeds. On a training site the number is the rest of that site. On a held-out site the number is the score models got there. An open circle has no finished run yet. Basel, Dornach, and Arlesheim sit in one neighbourhood, so those cards are pulled into the open part of the map.

Beside the map, two panels show the share of labelled plant pixels named correctly and the share called background. Under those is the share named as another plant. Those three shares add to 1, and empty ground is not counted. The site-by-plant panel is held-out species F1, best plant on top and best site on the left. The bottom row places each training site beside the held-out sites. The first number is species F1 and the second is the gap between seed 42 and seed 43. A grey cell means that plant does not occur on the site named by the column.

In [ ]:
import runpy
runpy.run_path(str(repo / "notebooks" / "plot_few_shot_site_map.py"))
from IPython.display import Image, display
display(Image(filename=str(FIG_DIR / "run_plan.png")))
display(Image(filename=str(FIG_DIR / "site_map_overview.png")))